In [ ]:
import pyspark
from pyspark.sql import SparkSession

import os
os.environ["JAVA_HOME"] = "/opt/homebrew/Cellar/openjdk@11/11.0.26/libexec/openjdk.jdk/Contents/Home"

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

df = spark.read \
    .option("header", "true") \
    .parquet('yellow_tripdata_2024-10.parquet')

df.show()

In [ ]:
df = spark.read.parquet('yellow_trip_data_2024_10_pq')

In [12]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee']

In [ ]:
df.createOrReplaceTempView('trip_data_2024_10')

In [ ]:
spark.sql("""
SELECT
    count(tpep_pickup_datetime)
FROM
    trip_data_2024_10
where 
    date_trunc('day', tpep_pickup_datetime) = '2024-10-15'
""").show()

In [ ]:
spark.sql("""
SELECT Max((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600 )
FROM trip_data_2024_10
          order by 1 desc
""").show()

In [ ]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

In [ ]:
df_zone = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

In [17]:
df_zone = df_zone.withColumnRenamed("LocationID", "PULocationID")


In [18]:
df_zone.show()

+------------+-------------+--------------------+------------+
|PULocationID|      Borough|                Zone|service_zone|
+------------+-------------+--------------------+------------+
|           1|          EWR|      Newark Airport|         EWR|
|           2|       Queens|         Jamaica Bay|   Boro Zone|
|           3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|           4|    Manhattan|       Alphabet City| Yellow Zone|
|           5|Staten Island|       Arden Heights|   Boro Zone|
|           6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|           7|       Queens|             Astoria|   Boro Zone|
|           8|       Queens|        Astoria Park|   Boro Zone|
|           9|       Queens|          Auburndale|   Boro Zone|
|          10|       Queens|        Baisley Park|   Boro Zone|
|          11|     Brooklyn|          Bath Beach|   Boro Zone|
|          12|    Manhattan|        Battery Park| Yellow Zone|
|          13|    Manhattan|   Battery Park City| Yello

In [21]:
df_joined = df.join(df_zone,on='PULocationID', how='outer')

In [22]:
df_joined.show()

+------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+---------+------------+------------+
|PULocationID|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|  Borough|        Zone|service_zone|
+------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+---------+------------+------------+
|          12|       2| 2024-10-02 13:46:19|  2024-10-02 14:01:31|             

In [23]:
df_joined.createOrReplaceTempView('joined')

In [25]:
spark.sql("""
SELECT
    Zone,
    count(tpep_pickup_datetime) as count
FROM
    joined
group by Zone
Order by count asc
""").show()

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|    Great Kills Park|    0|
|     Freshkills Park|    0|
|Governor's Island...|    1|
|       Arden Heights|    2|
|       Rikers Island|    2|
| Green-Wood Cemetery|    3|
|         Jamaica Bay|    3|
|   Rossville/Woodrow|    4|
|Charleston/Totten...|    4|
|Eltingville/Annad...|    4|
|       West Brighton|    4|
|       Port Richmond|    4|
|        Crotona Park|    6|
|         Great Kills|    6|
|     Mariners Harbor|    7|
|Heartland Village...|    7|
|Saint George/New ...|    9|
|             Oakwood|    9|
|       Broad Channel|   10|
|New Dorp/Midland ...|   10|
+--------------------+-----+
only showing top 20 rows

